In [20]:
from pathlib import Path
import json

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler


In [21]:
# Paths
ROOT = Path.cwd().parent             # 04_Models -> project root
SEQ_DIR = ROOT / "03_Sequences"

X_train_path = SEQ_DIR / "X_train.npy"
y_train_path = SEQ_DIR / "y_train.npy"
X_test_path  = SEQ_DIR / "X_test.npy"
y_test_path  = SEQ_DIR / "y_test.npy"
meta_path    = SEQ_DIR / "meta.json"

print("Using:")
print("  ", X_train_path)
print("  ", y_train_path)
print("  ", X_test_path)
print("  ", y_test_path)
print("  ", meta_path)

# Load arrays
X_train = np.load(X_train_path)
y_train = np.load(y_train_path)
X_test  = np.load(X_test_path)
y_test  = np.load(y_test_path)

with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("meta:", meta)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


Using:
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/X_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/y_train.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/X_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/y_test.npy
   /Users/simonhinterreiter/Library/CloudStorage/Dropbox/06 - Coding/01 - Local Git Repos/02 - Mac/STUDY.DeepLearning/Project/03_Sequences/meta.json
meta: {'seq_len': 60, 'feature_cols': ['log_ret_1m', 'log_ret_5m', 'log_ret_15m', 'vol_5m', 'vol_15m', 'range_pct_1m', 'range_pct_5m', 'body_pct', 'upper_wick_pct', 'lower_wick_pct', 'body_norm', 'volume_ratio_5m', 'volume_ratio_15m', 'log_re

In [22]:
# -----------------------------
# Validation split from training
# -----------------------------
val_ratio = 0.1
n_train = X_train.shape[0]
val_size = int(n_train * val_ratio)

X_train_final = X_train[:-val_size]
y_train_final = y_train[:-val_size]

X_val = X_train[-val_size:]
y_val = y_train[-val_size:]

print("Train final:", X_train_final.shape, y_train_final.shape)
print("Val        :", X_val.shape, y_val.shape)
print("Test       :", X_test.shape, y_test.shape)


Train final: (1704446, 60, 15) (1704446,)
Val        : (189382, 60, 15) (189382,)
Test       : (473458, 60, 15) (473458,)


In [23]:
# -----------------------------
# Scaler
# -----------------------------

from sklearn.preprocessing import StandardScaler

num_features = X_train_final.shape[2]

scaler = StandardScaler()

# Fit on training data (flatten time dimension)
X_train_2d = X_train_final.reshape(-1, num_features)
scaler.fit(X_train_2d)

def apply_scaler(X):
    n, seq_len, nf = X.shape
    X_2d = X.reshape(-1, nf)
    X_scaled_2d = scaler.transform(X_2d)
    return X_scaled_2d.reshape(n, seq_len, nf)

X_train_scaled = apply_scaler(X_train_final)
X_val_scaled   = apply_scaler(X_val)
X_test_scaled  = apply_scaler(X_test)

print("Scaled shapes:")
print("  train:", X_train_scaled.shape)
print("  val  :", X_val_scaled.shape)
print("  test :", X_test_scaled.shape)


Scaled shapes:
  train: (1704446, 60, 15)
  val  : (189382, 60, 15)
  test : (473458, 60, 15)


In [24]:
# -----------------------------
# Wrap data in PyTorch Dataset
# -----------------------------

class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        # X: (N, seq_len, num_features)
        # y: (N,)
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()  # float for BCEWithLogitsLoss

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 16

train_ds = TimeSeriesDataset(X_train_scaled, y_train_final)
val_ds   = TimeSeriesDataset(X_val_scaled,   y_val)
test_ds  = TimeSeriesDataset(X_test_scaled,  y_test)

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)

len(train_ds), len(val_ds), len(test_ds)


(1704446, 189382, 473458)

In [25]:
# -----------------------------
# Model Spec
# -----------------------------

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS (Apple Silicon GPU)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

seq_len = X_train_scaled.shape[1]
num_features = X_train_scaled.shape[2]


class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, 1)  # 1 output logit

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (h_n, c_n) = self.lstm(x)
        # h_n: (num_layers, batch, hidden_size)
        h_last = h_n[-1]  # (batch, hidden_size)
        logits = self.fc(h_last)  # (batch, 1)
        return logits.squeeze(1)  # (batch,)


model = LSTMClassifier(
    input_size=num_features, hidden_size=64, num_layers=2, dropout=0.1
)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(model)

Using device: MPS (Apple Silicon GPU)
LSTMClassifier(
  (lstm): LSTM(15, 64, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


In [45]:
# -----------------------------
# Training Loop
# -----------------------------
import time
from tqdm.notebook import tqdm
from torch import nn
import torch

def run_epoch(model, loader, criterion, device, optimizer=None):
    """
    If optimizer is provided -> training mode.
    If optimizer is None    -> evaluation mode.
    Returns (avg_loss, avg_accuracy).
    """
    if optimizer is None:
        model.eval()
    else:
        model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    print(f"[DEBUG]  loader has {len(loader)} batches")

    mode = "Train" if optimizer is not None else "Eval"

    # Use notebook tqdm
    progress_bar = tqdm(loader, desc=mode, leave=False)

    for X_batch, y_batch in progress_bar:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        if optimizer is not None:
            optimizer.zero_grad()

        # Forward
        logits = model(X_batch)           # (batch,)

        loss = criterion(logits, y_batch) # y_batch is float (0 or 1)

        # Backward + step only in training
        if optimizer is not None:
            loss.backward()
            optimizer.step()

        # Predictions for accuracy
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            correct = (preds == y_batch).sum().item()

        batch_size_curr = y_batch.size(0)
        total_loss += loss.item() * batch_size_curr
        total_correct += correct
        total_samples += batch_size_curr

    avg_loss = total_loss / total_samples
    avg_acc = total_correct / total_samples
    return avg_loss, avg_acc

def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)
    return mins, secs




In [46]:
import torch.optim as optim

def train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=5,
    lr=1e-3,
    model_path="best_lstm_model.pt",
):
    model = model.to(device)

    # Binary classification -> BCEWithLogitsLoss
    criterion = nn.BCEWithLogitsLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_valid_loss = float("inf")

    train_losses = []
    train_accuracies = []
    valid_losses = []
    valid_accuracies = []

    for epoch in tqdm(range(epochs)):
        print(f"\n[DEBUG] Starting epoch {epoch+1}/{epochs}")

        start_time = time.monotonic()

        # ---- TRAIN PHASE ----
        print("[DEBUG]  Running train_epoch...")
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, device, optimizer=optimizer
        )
        print("[DEBUG]  Finished train_epoch.")

        # ---- VALIDATION PHASE ----
        print("[DEBUG]  Running validation...")
        valid_loss, valid_acc = run_epoch(
            model, val_loader, criterion, device, optimizer=None
        )
        print("[DEBUG]  Finished validation.")

        # ---- SAVE BEST MODEL ----
        if valid_loss < best_valid_loss:
            print(f"[DEBUG]  Saving new best model (val_loss={valid_loss:.4f})")
            best_valid_loss = valid_loss
            torch.save(model.state_dict(), model_path)


        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)

        print(f"Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s")
        print(
            f"\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%"
        )
        print(
            f"\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%"
        )

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_acc)

    print("\n[DEBUG] Training completed.\n")

    return {
        "train_losses": train_losses,
        "train_accuracies": train_accuracies,
        "valid_losses": valid_losses,
        "valid_accuracies": valid_accuracies,
        "best_model_path": model_path,
    }


In [47]:
print("device:", device)
print("len(train_ds):", len(train_ds))
print("len(train_loader):", len(train_loader))
print("batch_size:", batch_size)


history = train_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=1,
    lr=1e-3,
    model_path="best_lstm_model.pt",   # or Path("04_Models/best_lstm_model.pt")
)


device: mps
len(train_ds): 1704446
len(train_loader): 106528
batch_size: 16


  0%|          | 0/1 [00:00<?, ?it/s]


[DEBUG] Starting epoch 1/1
[DEBUG]  Running train_epoch...
[DEBUG]  loader has 106528 batches


Train:   0%|          | 0/106528 [00:00<?, ?it/s]

[DEBUG]  Finished train_epoch.
[DEBUG]  Running validation...
[DEBUG]  loader has 11837 batches


Eval:   0%|          | 0/11837 [00:00<?, ?it/s]

[DEBUG]  Finished validation.
[DEBUG]  Saving new best model (val_loss=0.4862)
Epoch: 01 | Time: 18m 8s
	Train Loss: 0.498 | Train Acc: 72.87%
	 Val. Loss: 0.486 |  Val. Acc: 73.54%

[DEBUG] Training completed.



In [52]:
# -----------------------------
# Load best model
# -----------------------------

best_model = LSTMClassifier(input_size=num_features, hidden_size=64, num_layers=2, dropout=0.1)
best_model.load_state_dict(torch.load("best_lstm_model.pt", map_location=device))
best_model.to(device)

LSTMClassifier(
  (lstm): LSTM(15, 64, num_layers=2, batch_first=True, dropout=0.1)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)

In [54]:
# -----------------------------
# Evaluation on Test Set
# -----------------------------

test_loss, test_acc = run_epoch(
    best_model, test_loader, nn.BCEWithLogitsLoss().to(device), device, optimizer=None
)
print(f"Test loss: {test_loss:.4f}, Test acc: {test_acc:.4f}")


[DEBUG]  loader has 29592 batches


Eval:   0%|          | 0/29592 [00:00<?, ?it/s]

Test loss: 0.4916, Test acc: 0.7321


In [56]:
# -----------------------------
# Accuracy per t_to_end_min
# -----------------------------
import pandas as pd

# Get the index of t_to_end_min in the feature columns
t_to_end_min_idx = meta["feature_cols"].index("t_to_end_min")
print(f"t_to_end_min is at feature index: {t_to_end_min_idx}")

# Extract t_to_end_min from the UNSCALED test data (last timestep of each sequence)
# We use X_test (unscaled) to get the original minute values
t_to_end_min_values = X_test[:, -1, t_to_end_min_idx]  # shape: (num_test,)

print(f"Unique t_to_end_min values: {np.unique(t_to_end_min_values)}")

# Get predictions from the best model
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        logits = best_model(X_batch)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Create DataFrame for analysis
df_results = pd.DataFrame({
    "t_to_end_min": t_to_end_min_values,
    "y_true": all_labels,
    "y_pred": all_preds,
})

# Compute confusion columns
df_results["tp"] = ((df_results.y_true == 1) & (df_results.y_pred == 1)).astype(int)
df_results["fp"] = ((df_results.y_true == 0) & (df_results.y_pred == 1)).astype(int)
df_results["tn"] = ((df_results.y_true == 0) & (df_results.y_pred == 0)).astype(int)
df_results["fn"] = ((df_results.y_true == 1) & (df_results.y_pred == 0)).astype(int)
df_results["correct"] = (df_results["tp"] + df_results["tn"]).astype(int)

# Group by minute offset
stats = df_results.groupby("t_to_end_min").agg(
    count=("correct", "count"),
    tp=("tp", "sum"),
    fp=("fp", "sum"),
    tn=("tn", "sum"),
    fn=("fn", "sum"),
    accuracy=("correct", "mean"),
).reset_index()

stats["accuracy_pct"] = stats["accuracy"] * 100

# Print the results
print("\n" + "="*70)
print("ACCURACY + CONFUSION METRICS PER t_to_end_min")
print("="*70)
print(stats.to_string(index=False))
print("="*70)

print(f"\nOverall Test Accuracy: {all_labels.mean():.4f} (baseline if always predicting 1)")
print(f"Model Test Accuracy:  {(all_preds == all_labels).mean():.4f}")

t_to_end_min is at feature index: 14
Unique t_to_end_min values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15.]

ACCURACY + CONFUSION METRICS PER t_to_end_min
 t_to_end_min  count    tp   fp    tn   fn  accuracy  accuracy_pct
          1.0  31564  7995 7585  8238 7746  0.514288     51.428843
          2.0  31564 14795  894 14929  946  0.941706     94.170574
          3.0  31564 14347 1609 14214 1394  0.904860     90.485997
          4.0  31564 13925 2069 13754 1816  0.876917     87.691674
          5.0  31564 13495 2568 13255 2246  0.847484     84.748448
          6.0  31564 13093 3047 12776 2648  0.819573     81.957293
          7.0  31564 12821 3424 12399 2920  0.799012     79.901153
          8.0  31564 12428 3888 11935 3313  0.771860     77.186035
          9.0  31564 11973 4402 11421 3768  0.741161     74.116082
         10.0  31564 11517 4977 10846 4224  0.708497     70.849702
         11.0  31564 11264 5605 10218 4477  0.680585     68.058548
         12.0  31564 1